# Sagert EOS: Neutron Star Computational Implementation

This notebook develops the computational implementation of the physics explored in `sagert_neutron_star_physics.ipynb`, following Sagert et al., *Compact Stars for Undergraduates*.

## Aim

To translate the physical equations developed in the accompanying physics notebook into Python, building and testing the numerical methods required to investigate neutron star equations of state and stellar structure.

### Numerical verification

Numerical calculations will be checked against the corresponding analytical expressions, limiting behaviour and results presented in Sagert et al., where applicable.

Discrepancies, numerical limitations and assumptions will be investigated and documented throughout development.

In [1]:
# Import the libraries

from scipy import constants  
import numpy as np               
import matplotlib.pyplot as plt 
from scipy.integrate import solve_ivp       

In [2]:
# Define the physical constants in SI units

hbar = constants.hbar                   # Reduced Planck constant, J s
pi = constants.pi                          # pi
m_n = constants.m_n                  # Neutron mass, kg
c = constants.c                      # Speed of light, m/s
G = constants.G                      # Gravitational constant, N m**2/kg**2


In [3]:
# Calculate the neutron polytropic constant using Sagert Eq. (73)

k_nrel = hbar**2/(15*pi**2*m_n)*(3*pi**2/(m_n*c**2))**(5/3)

In [4]:
# Define the nonrelativistic polytropic exponent from Sagert Eq. (72)

gamma = 5/3


In [5]:
# Define the neutron degeneracy pressure using Sagert Eq. (72)
# Converts energy density (J/m^3) into pressure (Pa)

def non_relativistic_pressure(epsilon):
    return k_nrel*epsilon**gamma

In [6]:
# Invert the polytropic EOS to calculate energy density from pressure
# Sagert Eq. (72)
# p: pressure in Pa
# K: polytropic constant
# gamma: polytropic exponent
# Returns energy density in J/m^3

def polytropic_energy_density(p, k_nrel, gamma):
    return (p/k_nrel)**(1/gamma)

In [ ]:
# Define a test energy density in SI units
# Used to verify the neutron polytropic EOS

epsilon = 1e33

In [ ]:
# Calculate the pressure from the test energy density
# Uses the nonrelativistic neutron EOS from Sagert Eq. (72)

p = non_relativistic_pressure(epsilon)

In [11]:
# Recover energy density from the calculated pressure
# Checks that the forward and inverse EOS give consistent results

polytropic_energy_density(p, k_nrel, gamma)

1.0000000000000005e+33

In [15]:
# Calculate the rate of change of enclosed mass using Sagert Eq. (7)
# Determines how much mass is added as we move outwards through the star
# r: radius in metres
# epsilon: energy density in J/m^3
# Returns dm/dr in kg/m

def mass_gradient(r, epsilon):
    return 4*pi*r**2*epsilon/c**2

In [14]:
# Calculate the Newtonian pressure gradient using Sagert Eq. (8)
# Determines how pressure changes with radius under gravity
# r: radius in metres
# epsilon: energy density in J/m^3
# m: enclosed mass in kg
# Returns dp/dr in Pa/m

def pressure_gradient(r, epsilon, m):
    return -G*epsilon*m/(c**2*r**2)

In [13]:
# Combine Sagert Eqs. (7) and (8) using the polytropic EOS
# Calculates the mass and pressure gradients needed by the numerical solver
# y: current state [m, p], containing enclosed mass (kg) and pressure (Pa)

def stellar_structure(r, y, k_nrel, gamma):
    m, p = y

    epsilon = polytropic_energy_density(max(p, 0.0), k_nrel, gamma)

    dm_dr = mass_gradient(r, epsilon)
    dp_dr = pressure_gradient(r, epsilon, m)

    return [dm_dr, dp_dr]


In [17]:
# Set the initial conditions for integrating the stellar structure equations
# Calculates the starting mass and pressure at a small radius near the centre
# p_c: central pressure in Pa
# r_c: starting radius in metres
# epsilon_c: central energy density calculated from the polytropic EOS
# m_c: initial enclosed mass, assuming approximately constant central density
# Returns [m_c, p_c] for the numerical solver

def initial_conditions(p_c, r_c, k_nrel, gamma):
    epsilon_c = polytropic_energy_density(p_c, k_nrel, gamma)

    m_c = (4*pi*r_c**3*epsilon_c)/(3*c**2)

    return [m_c, p_c]

In [18]:
# Define the stellar surface where pressure reaches zero and stop integration when the surface is reached

def stellar_surface(r, y, k_nrel, gamma):
    m, p = y
    return p

stellar_surface.terminal = True
stellar_surface.direction = -1

In [27]:
# Set the central conditions for a single neutron star model
# rho_c: chosen central mass density in kg/m^3
# epsilon_c: central energy density, approximated by rho_c*c**2
# p_c: central pressure calculated using the neutron polytropic EOS
# r_c: small positive starting radius to avoid the singularity at r = 0
# y_c: initial state [m_c, p_c] returned by initial_conditions
# Display the central pressure and initial state for verification

rho_c = 1e18
epsilon_c = rho_c*c**2


p_c = non_relativistic_pressure(epsilon_c)

# Begin one metre from the stellar centre
r_c = 1.0

# Initial enclosed mass and pressure
y_c = initial_conditions(p_c, r_c, k_nrel, gamma)

print("Central pressure:", p_c, "Pa")
print("Initial state:", y_c)

Central pressure: 5.38030568261923e+33 Pa
Initial state: [4.1887902047863926e+18, 5.38030568261923e+33]


In [29]:
# Integrate the Newtonian stellar structure equations using Sagert Eqs. (7) and (8)
# Calculates how enclosed mass and pressure change with radius
# stellar_structure: provides the mass and pressure gradients
# (r_c, 5e7): integration range in metres, starting near the centre
# y_c: initial state containing enclosed mass and central pressure
# args: passes k_nrel and gamma to the structure and surface functions
# events: stops integration when the stellar surface is reached (p = 0)
# rtol: relative error tolerance for the numerical solver
# atol: absolute error tolerances for mass (kg) and pressure (Pa)

solution = solve_ivp(
    stellar_structure,
    (r_c, 5e7),
    y_c,
    args=(k_nrel, gamma),
    events=stellar_surface,
    rtol=1e-8,
    atol=[1e15, 1e18]
)

print("Solver status:", solution.message)
print("Surface detected:", len(solution.t_events[0]) > 0)


Solver status: A termination event occurred.
Surface detected: True


In [ ]:
# Extract the stellar radius and enclosed mass at the surface (p = 0)
# t_events[0][0]: radius where the pressure first reaches zero, in metres
# y_events[0][0][0]: enclosed mass at that radius, in kg
# M_sun: solar mass in kg, used to express stellar mass in solar masses
# Convert the radius from metres to kilometres and mass from kg to solar masses

R = solution.t_events[0][0]
M = solution.y_events[0][0][0]

M_sun = 1.98847e30


print("Stellar radius:", R/1000, "km")
print("Stellar mass:", M/M_sun, "solar masses")


Stellar radius: 14.63204651358343 km
Stellar mass: 1.10155590936461 solar masses


In [34]:
# Check numerical convergence of the Newtonian neutron star model
# Repeat the integration with tighter relative and absolute tolerances
# Keep the initial conditions and stellar structure equations unchanged
# solution_fine: result of the integration using tighter tolerances
# R_fine and M_fine: radius and mass obtained from the refined solution
# Compare the refined results with the original mass and radius

solution_fine = solve_ivp(
    stellar_structure,
    (r_c, 5e7),
    y_c,
    args=(k_nrel, gamma),
    events=stellar_surface,
    rtol=1e-10,
    atol=[1e12, 1e7]
)

if solution_fine.success and len(solution_fine.t_events[0]) > 0:
    R_fine = solution_fine.t_events[0][0]
    M_fine = solution_fine.y_events[0][0][0]

    print("Radius difference (%):", abs(R_fine-R)/R*100)
    print("Mass difference (%):", abs(M_fine-M)/M*100)
else:
    print("Convergence test failed:", solution_fine.message)


Radius difference (%): 2.6662947347653882e-05
Mass difference (%): 2.1464375308132425e-07


In [ ]:
# Define a general Newtonian solver for the neutron polytropic model
# Calculates stellar mass and radius for any specified central density


def solve_star(rho_c, k_nrel, gamma):
    epsilon_c = rho_c*c**2
    p_c = k_nrel*epsilon_c**gamma

    r_c = 1.0
    y_c = initial_conditions(p_c, r_c, k_nrel, gamma)

    solution = solve_ivp(
        stellar_structure,
        (r_c, 5e7),
        y_c,
        args=(k_nrel, gamma),
        events=stellar_surface,
        rtol=1e-10,
          atol=[1e12, 1e7]
    )

    if not solution.success or len(solution.t_events[0]) == 0:
        raise RuntimeError("Stellar surface was not found")

    R = solution.t_events[0][0]
    M = solution.y_events[0][0][0]

    return R, M
